In [1]:
import re
import pandas as pd
import os
names = {
    "ISICData-binary-smECE_no_groupsize.csv": "Skin Lesion Class.",
    "FolkDataset-income-classify-CA-smECE_no_groupsize.csv": "Income Class.",
    "CivilComments-smECE_no_groupsize.csv": "Comment Toxicity Class.",
    "faceData-smECE_no_groupsize.csv": "Age Reg.",
    "FolkDataset-income-CA-smECE_no_groupsize.csv": "Income Reg.",
    "FolkDataset-travel_time-CA-smECE_no_groupsize.csv": "Travel Time Reg.",
}
lb=r"\\[-3pt]"
names = {k: f"\\begin{{tabular}}[c]{{@{{}}c@{{}}}} {lb.join(v.split())}  \\end{{tabular}}" for k, v in names.items()}
smece={}
for k, v in names.items():
    df = pd.read_csv(k,index_col=[0,1]).squeeze()
    smece[v] = df
smece_df=pd.DataFrame.from_dict(smece, orient='columns')
smece_df.index.names=["Method", "Discr."]



def format_mean_std(val):
    # Check if the value is a string or has the expected format
    if not isinstance(val, str) and not pd.isna(val):
        # Try to convert the value to string
        val = str(val)

    # Check if the value matches the pattern "mean ± std"
    pattern = r'([0-9.]+)\s*±\s*([0-9.]+)'
    match = re.match(pattern, val)

    if match:
        mean, std = match.groups()
        return f"\\begin{{tabular}}[c]{{@{{}}c@{{}}}} {mean} \\\\[-5pt] \\scriptsize{{±{std}}} \\end{{tabular}}"
    else:
        # Return the original value if it doesn't match the pattern
        return val


# Apply the function to every element in the DataFrame
# If all values have this format, you can apply it to the entire DataFrame
df_formatted = smece_df.map(format_mean_std)

latex_table = df_formatted.to_latex(index_names=False,          # Remove default index names
                                column_format='rc|cccccc',   # First two columns right-aligned, rest center
                                multirow=True,            # Allow multi-row cells
                                
                                )
# print(latex_table)

# Get the original LaTeX representation
pattern = r'\\toprule\n(.*?)\n(.*?)\\midrule'
match = re.search(pattern, latex_table, flags=re.DOTALL)

if match:
    # Get the first row with column headers
    header_row = match.group(1)

    modified_header = re.sub(
        r'^(\s*&\s*&)', 'Method & Discretization &', header_row)

    # Create the replacement with the modified header row
    replacement = '\\toprule\n' + modified_header + '\n\\midrule'

    modified_latex = re.sub(pattern, lambda m: replacement, latex_table, )
    modified_latex = modified_latex.replace(r"\multirow[t]{6}{*}", r"\multirow{10}{*}")
print(modified_latex)
    

\begin{tabular}{rc|cccccc}
\toprule
Method & Discretization & \begin{tabular}[c]{@{}c@{}} Skin\\[-3pt]Lesion\\[-3pt]Class.  \end{tabular} & \begin{tabular}[c]{@{}c@{}} Income\\[-3pt]Class.  \end{tabular} & \begin{tabular}[c]{@{}c@{}} Comment\\[-3pt]Toxicity\\[-3pt]Class.  \end{tabular} & \begin{tabular}[c]{@{}c@{}} Age\\[-3pt]Reg.  \end{tabular} & \begin{tabular}[c]{@{}c@{}} Income\\[-3pt]Reg.  \end{tabular} & \begin{tabular}[c]{@{}c@{}} Travel\\[-3pt]Time\\[-3pt]Reg.  \end{tabular} \\
\midrule
Uncalibrated Baseline & / & 290.77 & 281.81 & 119.28 & 111.04 & 155.13 & 72.38 \\
\cline{1-8}
Multiaccurate Baseline & / & \begin{tabular}[c]{@{}c@{}} 169.28 \\[-5pt] \scriptsize{±6.24} \end{tabular} & \begin{tabular}[c]{@{}c@{}} 94.05 \\[-5pt] \scriptsize{±5.71} \end{tabular} & \begin{tabular}[c]{@{}c@{}} 71.38 \\[-5pt] \scriptsize{±6.89} \end{tabular} & \begin{tabular}[c]{@{}c@{}} 47.54 \\[-5pt] \scriptsize{±0.71} \end{tabular} & \begin{tabular}[c]{@{}c@{}} 56.61 \\[-5pt] \scriptsize{±0.29} \e

In [2]:
# First, transpose the DataFrame
df_transposed = df_formatted.transpose()

# Convert the transposed DataFrame to LaTeX
# Note: After transposition, we need to adjust the column format

latex_table = df_transposed.to_latex(
    index=True,              # Keep index as it becomes the first column after transposition
    index_names=False,       # Remove default index names
    # Left-align first column, center others
    column_format='l' + 'c' * (df_transposed.shape[1]),
    multirow=True
)
print(latex_table)
# Adjust the regex pattern for the transposed structure
pattern = r'\\toprule\n(.*?)\n(.*?)\\midrule'
match = re.search(pattern, latex_table, flags=re.DOTALL)

if match:
    # Get the first row with column headers
    header_row = match.group(1)
    modified_latex = modified_latex.replace(r"Multiaccurate Baseline", r"MA")
    modified_latex = modified_latex.replace(r"Uncalibrated Baseline", r"UC")

# print(modified_latex)

\begin{tabular}{lccccccccccccccc}
\toprule
Method & Uncalibrated Baseline & Multiaccurate Baseline & \multicolumn{6}{r}{LSBoost} & \multicolumn{6}{r}{MCBoost} & Ours \\
Discr. & / & / & 10 & 20 & 30 & 50 & 75 & 100 & 10 & 20 & 30 & 50 & 75 & 100 & / \\
\midrule
\begin{tabular}[c]{@{}c@{}} Skin\\[-3pt]Lesion\\[-3pt]Class.  \end{tabular} & 290.77 & \begin{tabular}[c]{@{}c@{}} 169.28 \\[-5pt] \scriptsize{±6.24} \end{tabular} & \begin{tabular}[c]{@{}c@{}} 146.68 \\[-5pt] \scriptsize{±27.44} \end{tabular} & \begin{tabular}[c]{@{}c@{}} 159.31 \\[-5pt] \scriptsize{±29.24} \end{tabular} & \begin{tabular}[c]{@{}c@{}} 160.80 \\[-5pt] \scriptsize{±36.54} \end{tabular} & \begin{tabular}[c]{@{}c@{}} 147.87 \\[-5pt] \scriptsize{±30.32} \end{tabular} & \begin{tabular}[c]{@{}c@{}} 144.05 \\[-5pt] \scriptsize{±22.57} \end{tabular} & \begin{tabular}[c]{@{}c@{}} 149.77 \\[-5pt] \scriptsize{±26.00} \end{tabular} & \begin{tabular}[c]{@{}c@{}} 278.75 \\[-5pt] \scriptsize{±17.81} \end{tabular} & \begin{tabul

In [3]:
df_transposed

Method                                             Uncalibrated Baseline  \
Discr.                                                                 /   
\begin{tabular}[c]{@{}c@{}} Skin\\[-3pt]Lesion\...                290.77   
\begin{tabular}[c]{@{}c@{}} Income\\[-3pt]Class...                281.81   
\begin{tabular}[c]{@{}c@{}} Comment\\[-3pt]Toxi...                119.28   
\begin{tabular}[c]{@{}c@{}} Age\\[-3pt]Reg.  \e...                111.04   
\begin{tabular}[c]{@{}c@{}} Income\\[-3pt]Reg. ...                155.13   
\begin{tabular}[c]{@{}c@{}} Travel\\[-3pt]Time\...                 72.38   

Method                                                                         Multiaccurate Baseline  \
Discr.                                                                                              /   
\begin{tabular}[c]{@{}c@{}} Skin\\[-3pt]Lesion\...  \begin{tabular}[c]{@{}c@{}} 169.28 \\[-5pt] \s...   
\begin{tabular}[c]{@{}c@{}} Income\\[-3pt]Class...  \begin{tabular}[c]{@{}c@{}} 94.05 \\[-5pt] \sc...   
\begin{tabular}[c]{@{}c@{}} Comment\\[-3pt]Toxi...  \begin{tabular}[c]{@{}c@{}} 71.38 \\[-5pt] \sc...   
\begin{tabular}[c]{@{}c@{}} Age\\[-3pt]Reg.  \e...  \begin{tabular}[c]{@{}c@{}} 47.54 \\[-5pt] \sc...   
\begin{tabular}[c]{@{}c@{}} Income\\[-3pt]Reg. ...  \begin{tabular}[c]{@{}c@{}} 56.61 \\[-5pt] \sc...   
\begin{tabular}[c]{@{}c@{}} Travel\\[-3pt]Time\...  \begin{tabular}[c]{@{}c@{}} 34.94 \\[-5pt] \sc...   

Method                                                                                        LSBoost  \
Discr.                                                                                             10   
\begin{tabular}[c]{@{}c@{}} Skin\\[-3pt]Lesion\...  \begin{tabular}[c]{@{}c@{}} 146.68 \\[-5pt] \s...   
\begin{tabular}[c]{@{}c@{}} Income\\[-3pt]Class...  \begin{tabular}[c]{@{}c@{}} 102.06 \\[-5pt] \s...   
\begin{tabular}[c]{@{}c@{}} Comment\\[-3pt]Toxi...  \begin{tabular}[c]{@{}c@{}} 41.04 \\[-5pt] \sc...   
\begin{tabular}[c]{@{}c@{}} Age\\[-3pt]Reg.  \e...  \begin{tabular}[c]{@{}c@{}} 41.51 \\[-5pt] \sc...   
\begin{tabular}[c]{@{}c@{}} Income\\[-3pt]Reg. ...  \begin{tabular}[c]{@{}c@{}} 72.29 \\[-5pt] \sc...   
\begin{tabular}[c]{@{}c@{}} Travel\\[-3pt]Time\...  \begin{tabular}[c]{@{}c@{}} 58.47 \\[-5pt] \sc...   

Method                                                                                                 \
Discr.                                                                                             20   
\begin{tabular}[c]{@{}c@{}} Skin\\[-3pt]Lesion\...  \begin{tabular}[c]{@{}c@{}} 159.31 \\[-5pt] \s...   
\begin{tabular}[c]{@{}c@{}} Income\\[-3pt]Class...  \begin{tabular}[c]{@{}c@{}} 102.57 \\[-5pt] \s...   
\begin{tabular}[c]{@{}c@{}} Comment\\[-3pt]Toxi...  \begin{tabular}[c]{@{}c@{}} 39.39 \\[-5pt] \sc...   
\begin{tabular}[c]{@{}c@{}} Age\\[-3pt]Reg.  \e...  \begin{tabular}[c]{@{}c@{}} 23.29 \\[-5pt] \sc...   
\begin{tabular}[c]{@{}c@{}} Income\\[-3pt]Reg. ...  \begin{tabular}[c]{@{}c@{}} 73.82 \\[-5pt] \sc...   
\begin{tabular}[c]{@{}c@{}} Travel\\[-3pt]Time\...  \begin{tabular}[c]{@{}c@{}} 57.75 \\[-5pt] \sc...   

Method                                                                                                 \
Discr.                                                                                             30   
\begin{tabular}[c]{@{}c@{}} Skin\\[-3pt]Lesion\...  \begin{tabular}[c]{@{}c@{}} 160.80 \\[-5pt] \s...   
\begin{tabular}[c]{@{}c@{}} Income\\[-3pt]Class...  \begin{tabular}[c]{@{}c@{}} 90.17 \\[-5pt] \sc...   
\begin{tabular}[c]{@{}c@{}} Comment\\[-3pt]Toxi...  \begin{tabular}[c]{@{}c@{}} 36.61 \\[-5pt] \sc...   
\begin{tabular}[c]{@{}c@{}} Age\\[-3pt]Reg.  \e...  \begin{tabular}[c]{@{}c@{}} 20.25 \\[-5pt] \sc...   
\begin{tabular}[c]{@{}c@{}} Income\\[-3pt]Reg. ...  \begin{tabular}[c]{@{}c@{}} 80.03 \\[-5pt] \sc...   
\begin{tabular}[c]{@{}c@{}} Travel\\[-3pt]Time\...  \begin{tabular}[c]{@{}c@{}} 54.69 \\[-5pt] \sc...   

Method                     

In [4]:
smece_df

\begin{tabular}[c]{@{}c@{}} Skin\\[-3pt]Lesion\\[-3pt]Class.  \end{tabular}  \
Method                 Discr.                                                                               
Uncalibrated Baseline  /                                                  290.77                            
Multiaccurate Baseline /                                           169.28 ± 6.24                            
LSBoost                10                                         146.68 ± 27.44                            
                       20                                         159.31 ± 29.24                            
                       30                                         160.80 ± 36.54                            
                       50                                         147.87 ± 30.32                            
                       75                                         144.05 ± 22.57                            
                       100                                        149.77 ± 26.00                            
MCBoost                10                                         278.75 ± 17.81                            
                       20                                         263.71 ± 18.93                            
                       30                                         261.02 ± 26.77                            
                       50                                         252.76 ± 17.20                            
                       75                                         239.26 ± 22.52                            
                       100                                        259.49 ± 21.55                            
Ours                   /                                           89.26 ± 15.63                            

                              \begin{tabular}[c]{@{}c@{}} Income\\[-3pt]Class.  \end{tabular}  \
Method                 Discr.                                                                   
Uncalibrated Baseline  /                                                  281.81                
Multiaccurate Baseline /                                            94.05 ± 5.71                
LSBoost                10                                         102.06 ± 11.17                
                       20                                         102.57 ± 17.25                
                       30                                           90.17 ± 6.59                
                       50                                         102.08 ± 12.13                
                       75                                         108.26 ± 14.03                
                       100                                        124.94 ± 20.48                
MCBoost                10                                          259.83 ± 5.00                
                       20                                         212.19 ± 33.65                
                       30                                         186.57 ± 41.96                
                       50                                         182.07 ± 45.79                
                       75                                         193.78 ± 47.80                
                       100                                        198.78 ± 31.04                
Ours                   /                                           87.30 ± 25.94                

                              \begin{tabular}[c]{@{}c@{}} Comment\\[-3pt]Toxicity\\[-3pt]Class.  \end{tabular}  \
Method                 Discr.                                                                                    
Uncalibrated Baseline  /                                                  119.28                                 
Multiaccurate Baseline /                                            71.38 ± 6.89                                 
LSBoost                10                                          41.